# Template: Classificacao Multiclasse (3 Classes)

Este notebook segue a mesma linha de raciocinio do fluxo binario, adaptado para 3 classes.

Principais mudancas:
- Saida: Dense(3, softmax)
- Loss: sparse_categorical_crossentropy (para rotulos 0/1/2)
- Predicao final: argmax nas probabilidades

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

## STEP 1: Load Data

In [ ]:
# Substitua pelo seu caminho real
df = pd.read_csv("data/your_multiclass_dataset.csv")

# Nome da coluna alvo
TARGET_COL = "target"
df.head()

## STEP 2: Prepare Features and Target

In [ ]:
X = df.drop(columns=[TARGET_COL]).values
y = df[TARGET_COL].astype(int).values

classes = np.unique(y)
print("Detected classes:", classes)
if len(classes) != 3:
    raise ValueError(f"Expected exactly 3 classes, but found {len(classes)}: {classes}")

## STEP 3: Scale Features

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled.shape

## STEP 4: Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

## STEP 5: Build and Compile Model

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(3, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## STEP 6: Train Model

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
)

## STEP 7: Evaluate and Predict

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(probs, axis=1)

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix (3x3):")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

## Optional: One-hot Labels
Se seu alvo estiver em one-hot encoding, use categorical_crossentropy no lugar de sparse_categorical_crossentropy.